### RAG Pipeline - Data Ingestion to Vector DB pipeline

1 - Data ingestion

Read all the PDF files in the directory and convert it into document data structure

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

c:\Users\Nijitha\OneDrive\Documents\Projects\AgenticAI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def process_all_pdfs(pdf_directory):

    """Process all pdf files in a directory"""
    
    all_documents = []

    # get the pdf files in the directory
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print (f"Found {len(pdf_files)} PDF files to process")

    # loop every file in the pdf_files
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing Employee_Benefits_Guide.pdf
Loaded 5 pages

Processing Employee_Handbook.pdf
Loaded 5 pages

Processing Employee_Leave_Policy.pdf
Loaded 5 pages

Processing Library_Rules_and_Regulations.pdf
Loaded 5 pages

Total documents loaded: 20


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-17T10:15:36+00:00', 'author': 'Meridian Analytics Inc. — Human Resources', 'keywords': '', 'moddate': '2026-07-17T10:15:36+00:00', 'subject': 'Employee Benefits Guide', 'title': 'Employee Benefits Guide - Meridian Analytics Inc.', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Employee_Benefits_Guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Employee_Benefits_Guide.pdf', 'file_type': 'pdf'}, page_content='Meridian Analytics Inc. — Confidential Internal Document\nHR-BEN-021 \x7f v5.2\nMeridian Analytics Inc.\nEmployee Benefits Guide\nPlan year 2026 — Health, financial, and wellness benefits overview\nDocument Code:\nHR-BEN-021\nVersion:\n5.2\nEffective Date:\nJanuary 1, 2026\nDocument Owner:\nDirector, Total Rewards\nApproved By:\nVP, People &amp; Culture\nClassification:\nInternal — All Employees\nMeridian Analytics Inc.\n1200 Ca

In [4]:
len(all_pdf_documents)

20

2 - Chunking

Text splitting get into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
def split_pdf_documents(documents, chunk_size = 1000, chunk_overlap = 200):

    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"\nContent: {split_docs[0].page_content[:200]}")
        print(f"\nMetadata: {split_docs[0].metadata}")

    return split_docs


In [10]:
chunks = split_pdf_documents(all_pdf_documents)
chunks

Split 20 documents into 46 chunks

Example chunk:

Content: Meridian Analytics Inc. — Confidential Internal Document
HR-BEN-021  v5.2
Meridian Analytics Inc.
Employee Benefits Guide
Plan year 2026 — Health, financial, and wellness benefits overview
Document C

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-17T10:15:36+00:00', 'author': 'Meridian Analytics Inc. — Human Resources', 'keywords': '', 'moddate': '2026-07-17T10:15:36+00:00', 'subject': 'Employee Benefits Guide', 'title': 'Employee Benefits Guide - Meridian Analytics Inc.', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Employee_Benefits_Guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Employee_Benefits_Guide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-17T10:15:36+00:00', 'author': 'Meridian Analytics Inc. — Human Resources', 'keywords': '', 'moddate': '2026-07-17T10:15:36+00:00', 'subject': 'Employee Benefits Guide', 'title': 'Employee Benefits Guide - Meridian Analytics Inc.', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Employee_Benefits_Guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Employee_Benefits_Guide.pdf', 'file_type': 'pdf'}, page_content='Meridian Analytics Inc. — Confidential Internal Document\nHR-BEN-021 \x7f v5.2\nMeridian Analytics Inc.\nEmployee Benefits Guide\nPlan year 2026 — Health, financial, and wellness benefits overview\nDocument Code:\nHR-BEN-021\nVersion:\n5.2\nEffective Date:\nJanuary 1, 2026\nDocument Owner:\nDirector, Total Rewards\nApproved By:\nVP, People &amp; Culture\nClassification:\nInternal — All Employees\nMeridian Analytics Inc.\n1200 Ca

3 - Embedding

Converting the chunks into vectors

In [11]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:

    """Handles document embedding generation using SentenceTransformer"""

    # Constructor to initialize the model and load the model
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):

        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()


    # load the model using Sentence transformer
    def _load_model(self):

        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimensions: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    # generating the embeddings for the chunks created
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1681.65it/s]


Model loaded successfully. Embedding Dimensions: 384


4 - Vector store

Storing the generated embeddings of the texts in the vector DB

In [22]:
class VectorStore:

    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
            
        """Initialize ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if(len(documents) != len(embeddings)):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                embeddings = embeddings_list,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [16]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-17T10:15:36+00:00', 'author': 'Meridian Analytics Inc. — Human Resources', 'keywords': '', 'moddate': '2026-07-17T10:15:36+00:00', 'subject': 'Employee Benefits Guide', 'title': 'Employee Benefits Guide - Meridian Analytics Inc.', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Employee_Benefits_Guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Employee_Benefits_Guide.pdf', 'file_type': 'pdf'}, page_content='Meridian Analytics Inc. — Confidential Internal Document\nHR-BEN-021 \x7f v5.2\nMeridian Analytics Inc.\nEmployee Benefits Guide\nPlan year 2026 — Health, financial, and wellness benefits overview\nDocument Code:\nHR-BEN-021\nVersion:\n5.2\nEffective Date:\nJanuary 1, 2026\nDocument Owner:\nDirector, Total Rewards\nApproved By:\nVP, People &amp; Culture\nClassification:\nInternal — All Employees\nMeridian Analytics Inc.\n1200 Ca

Extract the texts from the chunks

In [17]:
texts = [doc.page_content for doc in chunks]
texts

['Meridian Analytics Inc. — Confidential Internal Document\nHR-BEN-021 \x7f v5.2\nMeridian Analytics Inc.\nEmployee Benefits Guide\nPlan year 2026 — Health, financial, and wellness benefits overview\nDocument Code:\nHR-BEN-021\nVersion:\n5.2\nEffective Date:\nJanuary 1, 2026\nDocument Owner:\nDirector, Total Rewards\nApproved By:\nVP, People &amp; Culture\nClassification:\nInternal — All Employees\nMeridian Analytics Inc.\n1200 Cascade Parkway, Suite 400, Portland, OR 97204\nThis document is the property of Meridian Analytics Inc. and is intended solely for the use of its employees. Unauthorized\ndistribution outside the organization is prohibited.',
 "MERIDIAN ANALYTICS INC.\nEmployee Benefits Guide\nDocument HR-BEN-021  |  Effective January 1, 2026  |  Version 5.2\nConfidential — Internal Use Only\nPage 2\n1. Overview and Eligibility\nMeridian Analytics Inc. offers a comprehensive benefits package designed to support employees' physical health,\nfinancial security, and long-term well

Generate the embeddings for the texts

In [18]:
embeddings = embedding_manager.generate_embeddings(texts)

Generate embeddings for 46 texts...


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.25s/it]

Generated embeddings with shape: (46, 384)


In [20]:
embeddings

array([[-0.07141129,  0.08298692,  0.02846208, ..., -0.04895624,
        -0.0298386 , -0.07752697],
       [-0.05259478,  0.0745919 ,  0.0651643 , ..., -0.06415462,
        -0.07572231, -0.05668952],
       [-0.04947526,  0.05062218,  0.05432277, ..., -0.09809624,
        -0.00455728, -0.0039633 ],
       ...,
       [ 0.03956407,  0.02792422, -0.00483893, ..., -0.06323277,
        -0.06953701, -0.03328115],
       [-0.06095666,  0.00491792, -0.10064924, ..., -0.11529303,
         0.03619207, -0.03215944],
       [-0.03436221,  0.01414785,  0.02335028, ..., -0.05369975,
        -0.03818604, -0.08296299]], shape=(46, 384), dtype=float32)

Store the embeddings in the vector store

In [23]:
vector_store.add_documents(chunks, embeddings)

Adding 46 documents to vector store...
Successfully added 46 documents to vector store
Total documents in collection: 46


### Retrieval piepline from vector store

In [26]:
class RAGRetriever:

    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate the query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score 
                    similarity_score = 1 - distance

                    if similarity_score > score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")  
            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [27]:
rag_retriever.retrieve("What are the categories of leave?")

Retrieving documents for query: 'What are the categories of leave?'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.23it/s]

Generated embeddings with shape: (1, 384)


Retrieved 1 documents (after filtering)


[{'id': 'doc_7409b766_26',
  'content': 'third-party agencies are governed by the terms of their individual engagement agreements and are not covered by\nthis policy unless otherwise required by law.\nManagers are responsible for applying this policy consistently across their teams. Questions regarding\ninterpretation should be directed to the People & Culture (HR) team at hr-leave@meridiananalytics.com prior to\napproving or denying a request.\n2. Categories of Leave\nMeridian offers the following categories of leave. Eligibility, accrual, and approval requirements differ by category\nand are detailed in the sections that follow.\n■\nPaid Time Off (PTO) — discretionary time for rest, personal matters, and vacation.\n■\nSick Leave — time away due to illness, injury, or medical appointments for the employee or an immediate\nfamily member.\n■\nParental and Family Leave — leave following the birth, adoption, or foster placement of a child.\n■\nBereavement Leave — time away following the d

In [33]:
rag_retriever.retrieve("What is paid time off leave")

Retrieving documents for query: 'What is paid time off leave'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_96f0fec3_27',
  'content': "family member.\n■\nParental and Family Leave — leave following the birth, adoption, or foster placement of a child.\n■\nBereavement Leave — time away following the death of a family member.\n■\nJury Duty and Civic Leave — leave to fulfill civic obligations such as jury duty or voting.\n■\nMedical (FMLA) Leave — job-protected leave for serious health conditions, subject to eligibility.\n■\nUnpaid Personal Leave — leave granted at management discretion once other paid leave is exhausted.\n2.1 Paid Time Off (PTO)\nPTO is accrued on a bi-weekly basis according to length of service, as shown in Table 1. PTO accrual begins on\nan employee's first day of employment; however, new hires may not use accrued PTO during their first 90 days,\nexcept where required by state law.\nYears of Service\nAnnual Accrual\nBi-Weekly Accrual Rate\nMaximum Carryover\n0 – 2 years\n15 days\n4.62 hours\n5 days\n3 – 5 years\n20 days\n6.15 hours\n7 days\n6 – 10 years\n23 days

Integrating vector DB context pipeline with LLM output

Simple RAG pipeline with Groq LLM

In [34]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

True

Initialize GROQ LLM

In [39]:
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key = groq_api_key,
    model_name = "llama-3.3-70b-versatile",
    temperature = 0.1,
    max_tokens = 1024
)

Simple RAG function: retrieve content + generate response

In [40]:
def rag_simple(query, retriever, llm, top_k = 3):
    
    # retrieve the context
    results = retriever.retrieve(query, top_k = top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [41]:
answer = rag_simple("What are the categories of leave?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What are the categories of leave?'
Top K: 3, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


The categories of leave are: 
1. Paid Time Off (PTO)
2. Sick Leave
3. Parental and Family Leave
4. Bereavement Leave.


Enhanced RAG pipeline features

In [43]:
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):

    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)

    if not results:
        return{
            'answer': 'No relevant context found', 
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("what is Bereavement Leave?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is Bereavement Leave?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Bereavement Leave is up to 5 paid days for the death of an immediate family member and up to 2 paid days for the death of an extended family member.
Sources: [{'source': 'Employee_Leave_Policy.pdf', 'page': 3, 'score': 0.18785911798477173, 'preview': 'Requests should be submitted to People & Culture at least 30 days before the anticipated leave date where\nforeseeable.\n2.4 Bereavement Leave\nEmployees are granted up to five (5) paid days of bereavement leave upon the death of an immediate family\nmember (spouse, domestic partner, child, parent, sibl...'}]
Confidence: 0.18785911798477173
Context Preview: Requests should be submitted to People & Culture at least 30 days before the anticipated leave date where
foreseeable.
2.4 Bereavement Leave
Employees are granted up to five (5) paid days of bereavement leave upon the death of an immediate family
member (spouse, domestic partner, child, parent, sibl


Advanced RAG pipeline

In [44]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Bereavement Leave?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Bereavement Leave?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Requests should be submitted to People & Culture at least 30 days before the anticipated leave date where
foreseeable.
2.4 Bereavement Leave
Employees are granted up to fi

ve (5) paid days of bereavement leave upon the death of an immediate family
member (spouse, domestic partner, child, parent, sibling, grandparent, or grandchild) and up to two (2) paid days
for the death of an extended family member. Additional unpaid leave may be approved by a manager in
consultation with People & Culture.
2.5 Jury Duty and Civic Leave
Meridian provides paid leave for jury duty for the full duration of service, provided the employee furnishes a copy
of the jury summons and any court attendance verification to their manager. Employees are also granted up to
two (2) hours of paid time to vote in any public election if their work schedule does not otherwise provide sufficient
time outside of working hours.
2.6 Medical (FMLA) Leave

Question: what is Bereavement Leave?

Answer:

Final Answer: Bereavement Leave is up to 5 paid days for the death of an immediate family member and up to 2 paid days for the death of an extended family member.

Citations:
[1] Employee_Leave_Po

In [ ]:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Medical (FMLA) Leave", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Medical (FMLA) Leave'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.43it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
two (2) hours of paid time to vote in any public election if their work schedule does not o

therwise provide sufficient
time outside of working hours.
2.6 Medical (FMLA) Leave
Employees who have worked for Meridian for at least 12 months and logged a minimum of 1,250 hours in the
preceding 12 months are eligible for up to 12 weeks of unpaid, job-protected leave under the FMLA for a serious
health condition, to care for an immediate family member with a serious health condition, or for qualifying military
exigencies. Employees may elect, or Meridian may require, that accrued PTO and sick leave run concurrently with
FMLA leave.
3. Requesting Leave
All leave requests, regardless of category, must be submitted through the MeridianHR self-service portal at least
the number of business days specified below prior to the intended start date, except in emergency circumstances
where notice is not reasonably possible.
Leave Type
Minimum Advance Notice
Approval Authority
PTO (1–2 days)

Question: what is Medical (FMLA) Leave

Answer:

Final Answer: Up to 12 weeks of unpaid, job-protected